In [ ]:
# !pip install wordcloud
# !pip install chord
# !conda install pytorch torchvision torchaudio -c pytorch -y
# !pip install transformers
# !pip install plotapi
# !pip install happytransformer
# !pip install sumy

In [ ]:
#Data Manipulation
import pandas as pd
import datetime as dt
import numpy as np
import re

#Visualization
import matplotlib.pyplot as plt
# from plotapi import Chord
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
import matplotlib.pyplot as plt
from PIL import Image

#NLP Libraries
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
# import torch

#Text manipulation
import spacy
from spacy import displacy
from collections import Counter
import en_core_web_sm

# Uploading mp4 extracting transcript

In [ ]:
import sys
import time
import requests

filename = "2025 Q2 Group 7 Meet 2 - Shellisa.m4a"
 
def read_file(filename, chunk_size=5242880):
    with open(filename, 'rb') as _file:
        while True:
            data = _file.read(chunk_size)
            if not data:
                break
            yield data

headers = {'authorization': "fe4164b10bab43d19c3943e1b81f553b"}
response = requests.post('https://api.assemblyai.com/v2/upload',
                         headers=headers,
                         data=read_file(filename))

#this is needed for the next call below - 'audio url'
url = response.json()['upload_url']

#print(response.json())

In [ ]:
# #STEP 2
# #THIS CAN ALSO TAKE ONLINE HOSTED FILE WHERE URL CALL IS LOCATED

import requests
endpoint = "https://api.assemblyai.com/v2/transcript"


json = {
    "audio_url": f"{url}",
    'iab_categories': True, 
    "auto_chapters": True,
    "sentiment_analysis":True,
    "entity_detection": True,
      'auto_highlights': True,
      'speaker_labels': True
    ,
    'speakers_expected': 5
}

headers = {
    "authorization": "fe4164b10bab43d19c3943e1b81f553b",
    "content-type": "application/json"
}

response = requests.post(endpoint, json=json, headers=headers)
id = response.json()['id']

In [ ]:
# auto_highlights

In [ ]:
# STEP 3 - TO GET THE TRANSCRIPT

endpoint = f"https://api.assemblyai.com/v2/transcript/{id}"

headers = {
    "authorization": "fe4164b10bab43d19c3943e1b81f553b",

}

response = requests.get(endpoint,
                        headers=headers)

status = response.json()['status']

while status == 'queued' or status == 'processing':
    response = requests.get(endpoint, headers=headers)
    status = response.json()['status']

transcript_json = response.json()

### Headlines

In [ ]:
# for i, j in zip(transcript_json['chapters'], transcript_json['utterances']):
#     millis = i['start']
#     seconds=((millis/1000)%60)
#     seconds = int(seconds)
#     minutes=(millis/(1000*60))%60
#     minutes = int(minutes)
#     hours=(millis/(1000*60*60))%24

#     print ("%d:%d:%d" % (hours, minutes, seconds))
    
#     speaker = j['speaker']
#     headline = i['headline']
#     summary = i['summary']

#     # print(f"{headline}\n")

### Summary

In [ ]:
# for i, j in zip(transcript_json['chapters'], transcript_json['utterances']):
#     millis = i['start']
#     seconds=((millis/1000)%60)
#     seconds = int(seconds)
#     minutes=(millis/(1000*60))%60
#     minutes = int(minutes)
#     hours=(millis/(1000*60*60))%24

#     print ("%d:%d:%d" % (hours, minutes, seconds))
    
    
# #     seconds = ((i['start']/1000)%60)
# #     end_time = (i['end']/1000)
#     speaker = j['speaker']
#     headline = i['headline']
#     summary = i['summary']

# #     print(f"{round(start_time)} {round(end_time)}")
# #     print(f"Person {speaker} headline: {headline}\n")
# #     print(summary)
# #     print(f"{headline}\n")
#     print(f"{summary}\n")




In [ ]:
transcript_json['utterances']
df = pd.DataFrame.from_dict(transcript_json['utterances'])
df.to_csv(f"{filename}.csv")

In [ ]:
sentiment = pd.DataFrame.from_dict(transcript_json['sentiment_analysis_results'])
sentiment.to_csv(f"{filename}_sentiment.csv")
# sentiment

In [ ]:
df.to_csv(f"{filename}_sentiment.csv")
sentiment = pd.read_csv(f"{filename}_sentiment.csv")

In [ ]:
# df['speaker'].unique()

# Import CSV & Sentiment without re-running Assembly

In [ ]:
df = pd.read_csv('2025 Q2 Group 2 Meet 1 - Jake.m4a.csv')

In [ ]:
df

# <span style="color:#17499e"><b>CALL </span><span style="color:#2FC2F2">Analysis

# <span style="color:#17499e"><b>Data Visualization</span><span style="color:#2FC2F2">

In [ ]:
df['num_words_spoken'] = df['text'].str.split(" ")
df['num_words_spoken'] = df['num_words_spoken'].str.len()

In [ ]:
df['branch'] = df.groupby(['speaker'])['text'].transform(lambda x : ' '.join(x))
wordcloud = df[['speaker', 'branch']].drop_duplicates()
UNK = wordcloud[wordcloud['speaker'] == 'UNK'].index
wordcloud.drop(UNK, inplace= True)


participants = []
participant_transcript = []
for i,j in zip(wordcloud['branch'],wordcloud['speaker']):
        participant_transcript.append(i)
        participants.append(j)

In [ ]:
def num_words_spoken(df):
    words = df[['speaker', 'num_words_spoken']]
    words = words.groupby('speaker').sum('num_words_spoken')
    return words

In [ ]:
# Returns the df above
# num_words_spoken(df)

In [ ]:
import plotapi

plotapi.api_key("0fb0b4a5-5367-4dc7-b297-258549af4fd6")

In [ ]:
from plotapi import Terminus
import json
# Terminus.set_license("jason@relativ.ai", "PLOTAPI-P-0fb0b4a5-5367-4dc7-b297-258549af4fd6")

In [ ]:
# Dropping the unknown speaker category
words = df[['speaker', 'num_words_spoken']]
words = words.groupby('speaker').sum('num_words_spoken')
words['speaker'] = words.index
# words = words.drop(['UNK'])
# words

# PARTICIPATION BALANCE

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Assuming df is your original DataFrame with columns 'speaker' and 'num_words_spoken'

# 1. Aggregate the total words spoken per speaker
talk_time = df.groupby('speaker', as_index=False)['num_words_spoken'].sum()

# Optionally, remove the 'UNK' category if needed
# talk_time = talk_time[talk_time['speaker'] != 'UNK']

# 2. Calculate the total words spoken across all speakers
total_words = talk_time['num_words_spoken'].sum()

# 3. Calculate the percentage of conversation for each participant
talk_time['percentage'] = (talk_time['num_words_spoken'] / total_words) * 100

# 4. Sort speakers by percentage spoken for better visualization
talk_time_sorted = talk_time.sort_values('percentage', ascending=False)

# 5. Print each participant's percentage of the conversation
print("Percentage of Conversation per Participant:")
for _, row in talk_time_sorted.iterrows():
    print(f"{row['speaker']} spoke {row['percentage']:.2f}% of the conversation.")

# 6. Create a bar plot using Seaborn to visualize the percentages
plt.figure(figsize=(10, 6))
sns.barplot(data=talk_time_sorted, x='speaker', y='percentage', palette="viridis")
plt.xlabel("Speaker")
plt.ylabel("Percentage of Conversation (%)")
plt.title("Percentage of Conversation by Each Participant")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 7. Compute summary statistics: highest, lowest, and average percentage
max_percentage = talk_time['percentage'].max()
min_percentage = talk_time['percentage'].min()
avg_percentage = talk_time['percentage'].mean()

# 8. Print the summary statistics
print("\nSummary Statistics:")
print(f"Highest Percentage: {max_percentage:.2f}%")
print(f"Lowest Percentage: {min_percentage:.2f}%")
print(f"Average Percentage: {avg_percentage:.2f}%")


In [ ]:
# How do people related after the most talkative person is removed
# drop_unknown = without_most_talk[without_most_talk['speaker']]== 'UNK'
# without_most_talk = without_most_talk.drop()
without_most_talk = words.drop(words['num_words_spoken'].idxmax())

In [ ]:
without_most_talk['avg_words_without_most_talk'] = without_most_talk['num_words_spoken'].mean()

In [ ]:
without_most_talk['times_the_avg'] = without_most_talk['num_words_spoken']/without_most_talk['avg_words_without_most_talk']

In [ ]:
words['meeting'] = 'Meeting'
words = words.rename(columns={'meeting': 'source','speaker': 'target', 'num_words_spoken': 'value',})
words = words[['source', 'target', 'value']]
data = words.to_dict('records')
terminus = Terminus(data, 
             bundled=False, 
             pipe_alignment="top", 
             colors = 'set2',
             pixel_size=4,
             pixels_per_unit=20,
             title="<plotapi_count> words: Who spoke the most?").show()

In [ ]:
from plotapi import Chord
  


# Visualization code here


In [ ]:
from plotapi import Chord
Chord.api_key("0fb0b4a5-5367-4dc7-b297-258549af4fd6")

# Chord.set_license("jason@relativ.ai", "PLOTAPI-P-0fb0b4a5-5367-4dc7-b297-258549af4fd6")

# Chord.user = "jason@relativ.ai"
# Chord.key = "PLOTAPI-P-0fb0b4a5-5367-4dc7-b297-258549af4fd6"

    #Get chord diagram
links = df.loc[:, ('speaker', 'num_words_spoken')]
links['target'] = links['speaker'].shift(periods=-1)
UNK = links[links['speaker'] == 'UNK'].index
links.drop(UNK, inplace= True)
links['num_words_received'] = links['num_words_spoken'].shift(periods=-1)

names_label = pd.crosstab(links['speaker'], links['target'])
names = names_label.index.to_list()
matrix = names_label.values.tolist()

# Chord(matrix, names, conjunction="&", 
#       verb="interacted", 
#       noun="times",
#       popup_width=100).show()
# # Chord(matrix, names).show()
# # matrix

## Participation Balance ##

In [ ]:
import pandas as pd

# Assume `names_label` has been created as shown:
# names_label = pd.crosstab(links['speaker'], links['target'])

# Calculate the total interactions per participant (as a speaker)
interactions = names_label.sum(axis=1)

# Compute the group average of interactions
group_average = interactions.mean()

# Identify the participant with the most interactions
max_interactions = interactions.max()
participant_most = interactions.idxmax()

# Identify the participant with the least interactions
min_interactions = interactions.min()
participant_least = interactions.idxmin()

# Print the results
print("Interactions per Participant:")
print(interactions)
print("\nGroup Average Interactions: {:.2f}".format(group_average))
print("\nParticipant with Most Interactions: {} ({})".format(participant_most, max_interactions))
print("Participant with Least Interactions: {} ({})".format(participant_least, min_interactions))


### GINI CO Gini Coefficient

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# Prepare the chord diagram data
# -----------------------------
# Assume df is your DataFrame with columns 'speaker' and 'num_words_spoken'
links = df.loc[:, ('speaker', 'num_words_spoken')]

# Create a 'target' column by shifting the 'speaker' column by -1 (each row's speaker interacts with the next)
links['target'] = links['speaker'].shift(periods=-1)

# Remove rows where the speaker is 'UNK'
UNK = links[links['speaker'] == 'UNK'].index
links.drop(UNK, inplace=True)

# For completeness, create a 'num_words_received' column (not used in our Gini score calculation)
links['num_words_received'] = links['num_words_spoken'].shift(periods=-1)

# Create a crosstab: counts of interactions from each speaker to each target
names_label = pd.crosstab(links['speaker'], links['target'])

# -----------------------------
# Calculate Interactions and the Equal Interaction Score
# -----------------------------
# Compute total interactions initiated by each participant (summing across targets)
interactions = names_label.sum(axis=1)

def compute_gini_coefficient(values):
    """
    Compute the Gini coefficient for a numpy array of non-negative values.
    A Gini coefficient of 0 means perfect equality and 1 indicates maximum inequality.
    """
    # Convert input to a numpy array (float) and ensure it's one-dimensional.
    values = np.array(values, dtype=float).flatten()
    n = values.size

    # If the total number of interactions is zero, treat this as perfect equality.
    if np.sum(values) == 0:
        return 0.0

    # Sort the values in ascending order.
    sorted_values = np.sort(values)
    cumulative_values = np.cumsum(sorted_values)
    total = np.sum(sorted_values)

    # Calculate the Gini coefficient using the formula:
    # G = (n + 1 - 2*(sum(cumulative_values) / total)) / n
    gini = (n + 1 - 2 * (np.sum(cumulative_values) / total)) / n
    return gini

# Compute the Gini coefficient for the interaction counts.
gini = compute_gini_coefficient(interactions)

# Convert the Gini coefficient to an "Equal Interaction Score" where 0 Gini => 100 score
equal_interaction_score = (1 - gini) * 100

# -----------------------------
# Print the results
# -----------------------------
print("Interactions per participant:")
print(interactions)
print(f"\nGini Coefficient: {gini:.3f}")
print(f"Equal Interaction Score: {equal_interaction_score:.2f} (100 means perfectly equal interactions)")


In [ ]:


# # Your original code to generate links, drop UNK, and create names_label, names, and matrix
# links = df.loc[:, ('speaker', 'num_words_spoken')]
# links['target'] = links['speaker'].shift(periods=-1)
# UNK = links[links['speaker'] == 'UNK'].index
# links.drop(UNK, inplace=True)
# links['num_words_received'] = links['num_words_spoken'].shift(periods=-1)

# names_label = pd.crosstab(links['speaker'], links['target'])
# names = names_label.index.tolist()
# matrix = names_label.values.tolist()

# # Create a Chord plot
# chord_plot = Chord(matrix, names)

# # Show the Chord plot
# chord_plot.show()


In [ ]:
# matrix = [
#     [1, 0, 0, 0, 0, 0],
#     [0, 1, 2, 1, 1, 0],
#     [0, 0, 1, 0, 0, 1],
#     [1, 1, 0, 5, 3, 5],
#     [2, 1, 1, 4, 0, 1],
#     [0, 0, 4, 3, 0, 0],
# ]

# # matrix = [
# #     [1, 0, 0, 0, 0, 0],
# #     [0, 1, 2, 1, 1, 0],
# #     [0, 0, 1, 0, 0, 1],
# #     [1, 1, 0, 5, 3, 5],
# #     [1, 0, 7, 0, 5, 0],
# #     [2, 1, 1, 4, 0, 1],
# #     [0, 0, 4, 3, 0, 0],
# # ]

names = ["Jason", "Will", "Barbara-Shae", "Shellisa", "Tom"]


In [ ]:
# names = ['A','b', 'C', 'D', 'E']

In [ ]:
# Chord(matrix,names, 
#       curved_labels=True, 
#      reverse_gradients=True).show()

Chord(matrix, names, 
      padding=.5, 
      curved_labels=True, 
      conjunction="&", 
      verb="interacted", 
      noun="times",
      popup_width=100).show()

<!-- # <span style="color:#17499e"><b>Data Sales Language Metrics </span><span style="color:#2FC2F2">Concordance -->

In [ ]:
# starts = df['start'].to_numpy()
# ends = df['end'].to_numpy()

# end_to_starts = np.array([(starts[idx+1] - e) for idx,e in enumerate(ends) if idx < len(ends)-1])
# #     overlap_inds = np.where(end_to_starts<0)    

# df['talk_time_nanosec'] = ends - starts

# df['seconds'] = df['talk_time_nanosec']/1000

# df['num_words_spoken'] = df['text'].str.split(" ")
# df['num_words_spoken'] = df['num_words_spoken'].str.len()

# df['words_per_sec'] = df['num_words_spoken']/df['seconds']

# rate_of_speech = df.groupby(['speaker']).mean()
# rate_of_speech = rate_of_speech['words_per_sec']*60

# rate_of_speech

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# Calculate talk time in seconds
df['talk_time_sec'] = (df['end'] - df['start']) / 1000

# Calculate number of words spoken
df['num_words_spoken'] = df['text'].str.split().str.len()

# Calculate words per second
df['words_per_sec'] = df['num_words_spoken'] / df['talk_time_sec']

# Calculate words per minute (rate of speech)
df['words_per_min'] = df['words_per_sec'] * 60

# Group by speaker and calculate the mean words per minute
rate_of_speech = df.groupby('speaker')['words_per_min'].mean().reset_index()

# Let's assume you have N speakers
N = len(rate_of_speech['speaker'])
colors = plt.cm.viridis(np.linspace(0, 1, N))  # Generates an array of colors from the viridis colormap


# Plotting the bar graph
plt.figure(figsize=(10, 6))
plt.bar(rate_of_speech['speaker'], rate_of_speech['words_per_min'], color= colors)
plt.xlabel('Speaker')
plt.ylabel('Words per Minute')
plt.title('Rate of Speech (Words per Minute) by Speaker')
plt.xticks(rotation=45)
plt.grid(axis='y')

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Calculate talk time in seconds
df['talk_time_sec'] = (df['end'] - df['start']) / 1000

# Calculate number of words spoken and words per minute in one step
df['num_words_spoken'] = df['text'].str.split().str.len()
df['words_per_min'] = df['num_words_spoken'] / df['talk_time_sec'] * 60

# Group by speaker and calculate the mean words per minute
rate_of_speech = df.groupby('speaker', as_index=False)['words_per_min'].mean()

# Compute summary statistics
max_rate = rate_of_speech['words_per_min'].max()
min_rate = rate_of_speech['words_per_min'].min()
avg_rate = rate_of_speech['words_per_min'].mean()

print(f"Highest Rate of Speech: {max_rate:.2f} words per minute")
print(f"Lowest Rate of Speech: {min_rate:.2f} words per minute")
print(f"Average Rate of Speech: {avg_rate:.2f} words per minute")

# Generate a color palette for visualization
N = len(rate_of_speech)
colors = plt.cm.viridis(np.linspace(0, 1, N))

# Plotting the bar graph
plt.figure(figsize=(10, 6))
plt.bar(rate_of_speech['speaker'], rate_of_speech['words_per_min'], color=colors)
plt.xlabel('Speaker')
plt.ylabel('Words per Minute')
plt.title('Rate of Speech (Words per Minute) by Speaker')
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

# Assuming df is your DataFrame with 'speaker' and 'words_per_min' columns

# 1. Compute the average words per minute for each speaker
rate_of_speech = df.groupby('speaker')['words_per_min'].mean().reset_index()

# 2. Sort the results from highest to lowest
rate_of_speech_sorted = rate_of_speech.sort_values(by='words_per_min', ascending=False)

# 3. Print the sorted DataFrame
print("Rate of Speech per Participant:")
print(rate_of_speech_sorted)

# 4. Compute summary statistics: highest, lowest, and average rate of speech
max_rate = rate_of_speech['words_per_min'].max()
min_rate = rate_of_speech['words_per_min'].min()
avg_rate = rate_of_speech['words_per_min'].mean()

# 5. Print the summary statistics
print("\nSummary Statistics:")
print(f"Highest Rate of Speech: {max_rate:.2f} words per minute")
print(f"Lowest Rate of Speech: {min_rate:.2f} words per minute")
print(f"Average Rate of Speech: {avg_rate:.2f} words per minute")


# Ideas Generated ###

In [ ]:
import nltk
import matplotlib.pyplot as plt
import numpy as np

# Ensure required NLTK data is available
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

# Function to identify modal auxiliary verbs (POS tag starting with 'MD')
def is_modal(pos):
    return pos.startswith('MD')

# Dictionary to hold the counts of ideas for each participant
idea_counts = {}

# Process each participant's transcript
# Assume participant_transcript is a list of transcript strings and 
# participants is a corresponding list of participant names/IDs.
for text, speaker in zip(participant_transcript, participants):
    # Tokenize the text and tag parts of speech
    tokens = nltk.word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    
    # Extract modal auxiliary verbs as ideas
    modals = [word for word, pos in tagged if is_modal(pos)]
    
    # Count unique modal verbs as "ideas"
    num_ideas = len(set(modals))
    idea_counts[speaker] = num_ideas
    print(f"Participant {speaker} generated {num_ideas} ideas.")

# Calculate summary statistics
total_ideas = sum(idea_counts.values())
group_average = total_ideas / len(idea_counts)
max_ideas = max(idea_counts.values())
min_ideas = min(idea_counts.values())

print("\nSummary Statistics:")
print(f"Highest Ideas Generated: {max_ideas}")
print(f"Lowest Ideas Generated: {min_ideas}")
print(f"Group Average Ideas Generated: {group_average:.2f}")

# Prepare data for visualization
speakers = list(idea_counts.keys())
counts = list(idea_counts.values())

# Generate a color palette from the 'viridis' colormap
N = len(speakers)
colors = plt.cm.viridis(np.linspace(0, 1, N))

# Plot the bar graph
plt.figure(figsize=(10, 6))
plt.bar(speakers, counts, color=colors)
plt.xlabel('Participants')
plt.ylabel('Number of Ideas')
plt.title('Number of Ideas Generated by Each Participant')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:

# print(final_transcript_string)

In [ ]:
# my_list = ['apple', 'banana', 'cherry']
# # Join elements of the list with a comma and space
# result_str = ', '.join(my_list)
# print(result_str)  # Output: "apple, banana, cherry"

# # Join elements of the list without any delimiter
# result_str = ''.join(my_list)
# print(result_str)  # Output: "applebananacherry"

In [ ]:
# final_transcript_string = ' '.join(final_transcript)

In [ ]:
import matplotlib.pyplot as plt

# Sample participant transcripts (replace with your actual data)
# participant_transcript = [
#     "I like to play soccer with my friends.",
#     "Soccer is my favorite sport to watch on TV.",
#     "My friends and I often play board games after school."
# ]
# participants = ["A", "B", "C"]

# Dictionary to hold the unique word percentages for each participant
unique_word_percentages = {}

# Process each participant's transcript
for text, speaker in zip(participant_transcript, participants):
    words = text.split()
    unique_words = set(words)
    
    total_words = len(words)
    unique_words_count = len(unique_words)
    
    percentage_unique = (unique_words_count / total_words) * 100
    unique_word_percentages[speaker] = percentage_unique
    
    print(f"{unique_words_count} of Person {speaker}'s {total_words} words were unique to them. That's {percentage_unique:.2f}%")

# Prepare data for bar graph
speakers = list(unique_word_percentages.keys())
percentages = list(unique_word_percentages.values())

# Let's assume you have N speakers
N = len(rate_of_speech['speaker'])
colors = plt.cm.viridis(np.linspace(0, 1, N))  # Generates an array of colors from the viridis colormap


# Plotting the bar graph
plt.figure(figsize=(10, 6))
plt.bar(speakers, percentages, color=colors)
plt.xlabel('Participants')
plt.ylabel('Percentage of Unique Words')
plt.title('Percentage of Unique Words by Each Participant')
plt.ylim(0, 100)  # Set y-axis limits from 0 to 100%
plt.show()


In [ ]:
final_transcript = []
for i,j in zip(df['text'], df['speaker']):
    final_transcript.append(i)
    
listToStr = ' '.join(final_transcript)
res = re.sub(r'[^\w\s]', '', listToStr)
 
# storing string in the form of list of words
s = set(res.split(" "))

ques_in_convers = listToStr.count("?")
print(F"{ques_in_convers} total questions in the conversation.")

In [ ]:
# for i,j in zip(participants, participant_transcript):
#     num_ques = j.count("?")
#     print(f"Person {i} asked {num_ques} questions")
#     print()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


# Dictionary to hold the number of questions for each participant
question_counts = {}

# Process each participant's transcript
for participant, transcript in zip(participants, participant_transcript):
    # Count the number of question marks as an indicator of questions asked
    num_questions = transcript.count("?")
    question_counts[participant] = num_questions
    print(f"Participant {participant} asked {num_questions} questions.")

# Calculate summary statistics
total_questions = sum(question_counts.values())
num_participants = len(question_counts)
average_questions = total_questions / num_participants if num_participants > 0 else 0
max_questions = max(question_counts.values()) if num_participants > 0 else 0
min_questions = min(question_counts.values()) if num_participants > 0 else 0

print("\nSummary Statistics:")
print(f"Highest Questions Asked: {max_questions}")
print(f"Lowest Questions Asked: {min_questions}")
print(f"Average Questions Asked: {average_questions:.2f}")

# Prepare data for the bar graph
speakers = list(question_counts.keys())
questions = list(question_counts.values())

# Generate a color palette from the 'viridis' colormap based on the number of participants
N = len(speakers)
colors = plt.cm.viridis(np.linspace(0, 1, N))

# Plotting the bar graph
plt.figure(figsize=(10, 6))
plt.bar(speakers, questions, color=colors)
plt.xlabel('Participants')
plt.ylabel('Number of Questions Asked')
plt.title('Number of Questions Asked by Each Participant')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


***HEDGES (Sort of, Kind of, I think)***

In [ ]:
for i,j in zip(participants, participant_transcript):
    hedge_sort_of = j.count("sort of")
    hedge_kind_of = j.count("kind of")
    hedge_i_think = j.lower().count("I think")
    total_hedges = (hedge_sort_of) + (hedge_kind_of) +(hedge_i_think)
    print(f"Person {i} had {total_hedges} hedges. \n{hedge_sort_of} - Sort of: expresses uncertainty; possible lack of safety\n{hedge_kind_of} - Kind of: \n{hedge_i_think} - I think")
    print()

In [ ]:
# conversation = final_transcript_string

In [ ]:
transcript = ""
for text, speaker in zip(df['text'], df['speaker']):
    transcript += f"Person {speaker} - {text}\n\n"


In [ ]:
# transcript

In [ ]:
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.tag import pos_tag
import matplotlib.pyplot as plt

# Sample conversation data (replace with your actual conversation data)

# Function to extract nouns, verbs, and adjectives
def extract_pos(text):
    tokens = word_tokenize(text)
    tagged_tokens = pos_tag(tokens)
    
    nouns = [word for word, pos in tagged_tokens if pos.startswith('NN')]
    verbs = [word for word, pos in tagged_tokens if pos.startswith('VB')]
    adjectives = [word for word, pos in tagged_tokens if pos.startswith('JJ')]
    
    return nouns, verbs, adjectives

# Function to process transcript and extract/count for each person
def process_transcript(transcript):
    sentences = sent_tokenize(transcript)
    people = set()
    conversation_data = {}
    
    for sentence in sentences:
        if ' - ' in sentence:
            person = sentence.split(' - ')[0].strip()
            people.add(person)
            
            if person not in conversation_data:
                conversation_data[person] = {'nouns': [], 'verbs': [], 'adjectives': [], 'counts': {'nouns': 0, 'verbs': 0, 'adjectives': 0}}
            
            text = sentence.split(' - ')[1].strip()
            nouns, verbs, adjectives = extract_pos(text)
            
            conversation_data[person]['nouns'].extend(nouns)
            conversation_data[person]['verbs'].extend(verbs)
            conversation_data[person]['adjectives'].extend(adjectives)
            
            conversation_data[person]['counts']['nouns'] += len(nouns)
            conversation_data[person]['counts']['verbs'] += len(verbs)
            conversation_data[person]['counts']['adjectives'] += len(adjectives)
    
    return conversation_data

# Extract and count nouns, verbs, and adjectives for each person in the transcript
conversation_data = process_transcript(transcript)

# Data visualization using matplotlib
people = list(conversation_data.keys())
nouns_counts = [conversation_data[person]['counts']['nouns'] for person in people]
verbs_counts = [conversation_data[person]['counts']['verbs'] for person in people]
adjectives_counts = [conversation_data[person]['counts']['adjectives'] for person in people]

# Plotting
bar_width = 0.25
index = range(len(people))

plt.figure(figsize=(10, 6))
plt.bar(index, nouns_counts, width=bar_width, label='Nouns')
plt.bar(index, verbs_counts, width=bar_width, label='Verbs', bottom=nouns_counts)
plt.bar(index, adjectives_counts, width=bar_width, label='Adjectives', bottom=[nouns + verbs for nouns, verbs in zip(nouns_counts, verbs_counts)])

plt.xlabel('Person')
plt.ylabel('Counts')
plt.title('Counts of Nouns, Verbs, and Adjectives in Conversation')
plt.xticks(index, people)
plt.legend()

plt.tight_layout()
plt.show()


# Ums, Ah, uhs per person

In [ ]:
# for i,j in zip(participants, participant_transcript):
#     fillers_uh = j.count(" uh")
#     fillers_ah = j.count(" ah")
#     fillers_um = j.lower().count(" um")
#     total_fillers = (fillers_uh) + (fillers_ah) +(fillers_um)
#     print(f"Person {i} had {total_fillers} fillers. \n{fillers_uh} - Uh: expresses uncertainty; possible lack of safety\n{fillers_ah} - Ah: \n{fillers_um} - um")
#     print()

In [ ]:
# import nltk
# from nltk.tokenize import word_tokenize, sent_tokenize
# from nltk.tag import pos_tag
# import matplotlib.pyplot as plt

# #Convert transcript from list into string
# transcript = ""
# for text, speaker in zip(df['text'], df['speaker']):
#     transcript += f"Person {speaker} - {text}\n\n"

# # Function to extract nouns, verbs, and adjectives
# def extract_pos(text):
#     tokens = word_tokenize(text)
#     tagged_tokens = pos_tag(tokens)
    
#     nouns = [word for word, pos in tagged_tokens if pos.startswith('NN')]
#     verbs = [word for word, pos in tagged_tokens if pos.startswith('VB')]
#     adjectives = [word for word, pos in tagged_tokens if pos.startswith('JJ')]
    
#     return nouns, verbs, adjectives

# # Function to process conversation and extract/count for each person
# def process_conversation(conversation):
#     sentences = sent_tokenize(conversation)
#     people = set()
#     conversation_data = {}
    
#     for sentence in sentences:
#         if ' - ' in sentence:
#             person = sentence.split(':')[0].strip()
#             people.add(person)
            
#             if person not in conversation_data:
#                 conversation_data[person] = {'nouns': [], 'verbs': [], 'adjectives': [], 'counts': {'nouns': 0, 'verbs': 0, 'adjectives': 0}}
            
#             text = sentence.split(':')[1].strip()
#             nouns, verbs, adjectives = extract_pos(text)
            
#             conversation_data[person]['nouns'].extend(nouns)
#             conversation_data[person]['verbs'].extend(verbs)
#             conversation_data[person]['adjectives'].extend(adjectives)
            
#             conversation_data[person]['counts']['nouns'] += len(nouns)
#             conversation_data[person]['counts']['verbs'] += len(verbs)
#             conversation_data[person]['counts']['adjectives'] += len(adjectives)
    
#     return conversation_data

# # Extract and count nouns, verbs, and adjectives for each person in the conversation
# conversation_data = process_conversation(conversation)

# # Data visualization using matplotlib
# people = list(conversation_data.keys())
# nouns_counts = [conversation_data[person]['counts']['nouns'] for person in people]
# verbs_counts = [conversation_data[person]['counts']['verbs'] for person in people]
# adjectives_counts = [conversation_data[person]['counts']['adjectives'] for person in people]

# # Plotting
# bar_width = 0.25
# index = range(len(people))

# plt.figure(figsize=(10, 6))
# plt.bar(index, nouns_counts, width=bar_width, label='Nouns')
# plt.bar(index, verbs_counts, width=bar_width, label='Verbs', bottom=nouns_counts)
# plt.bar(index, adjectives_counts, width=bar_width, label='Adjectives', bottom=[nouns + verbs for nouns, verbs in zip(nouns_counts, verbs_counts)])

# plt.xlabel('Person')
# plt.ylabel('Counts')
# plt.title('Counts of Nouns, Verbs, and Adjectives in Conversation')
# plt.xticks(index, people)
# plt.legend()

# plt.tight_layout()
# plt.show()


# <span style="color:#17499e"><b> 🧮 WORD EFFICIENCY RATE

In [ ]:
text_str = f"""totally achieve this goal by next week.
Person B - I agree with you, it's an exciting project. We should focus on the key details and move quickly.
Person A - Absolutely! I feel so positive about the outcome. Let's make sure everyone stays focused.
Person B - Yes, focus is essential, and we need to communicate constant"""

In [ ]:
text_str

In [ ]:
NAME = "Jason"

In [ ]:
# for i, j in zip(participant_transcript, participants):
def idea_density (string, NAME):
    unique = set(string.split(" "))
    unique = len(unique)
    participant_words = len(text_str)
    
    fraction = unique/participant_words
    quotient = round(fraction*100)

    print(f"{NAME}, {unique} of your {participant_words} words were unique. That's {quotient}%.\nHow does {quotient}% compare to the list below?")

In [ ]:
idea_density(text_str, NAME)

### Chapters, Entities, Sentiment ###

In [ ]:
# transcript_json['chapters'][0]

In [ ]:
# transcript_json['entities'][0]

In [ ]:
# transcript_json['sentiment_analysis_results'][0]

### SENTIMENT from CSV

In [ ]:
# sentiment = pd.read_csv('Pete Buttigieg.mp3_sentiment.csv')
# sentiment['speaker'].value_counts()
# sentiment = sentiment[['sentiment', 'speaker', 'text']]

In [ ]:
# sentiment.head()

### Unlock with DF

In [ ]:
# sentiment = pd.DataFrame.from_dict(transcript_json['sentiment_analysis_results'])
# # sentiment.to_csv('Sentiment.csv')
# # # sentiment

# # sentiment = pd.read_csv('Sentiment.csv')

# #For the whole conversation
# sentiment['sentiment'].value_counts()

# #THIS IS WHAT WE WANT
# sentiment.groupby(['speaker'])['sentiment'].value_counts()

# # Group by speaker and sentiment, then count occurrences
# sentiment_counts = sentiment.groupby(['speaker', 'sentiment']).size().unstack(fill_value=0)

# # Plotting stacked bar graph
# plt.figure(figsize=(10, 6))
# sentiment_counts.plot(kind='bar', stacked=True, color=['red', '#FFFFED', 'green'])
# plt.xlabel('Speaker')
# plt.ylabel('Count')
# plt.title('Sentiment Distribution by Speaker')
# plt.legend(title='Sentiment', bbox_to_anchor=(1.05, 1), loc='upper left')

# plt.tight_layout()
# plt.show()

In [ ]:
# sentiment.groupby(['speaker']).agg(['max', 'min'])

In [ ]:
# sentiment.groupby(['speaker']).describe()

In [ ]:
# sentiment.groupby('speaker').sentiment.unique()

In [ ]:
df['branch'] = df.groupby(['speaker'])['text'].transform(lambda x : ' '.join(x))
wordcloud = df[['speaker', 'branch']].drop_duplicates()
UNK = wordcloud[wordcloud['speaker'] == 'UNK'].index
wordcloud.drop(UNK, inplace= True)


participants = []
participant_transcript = []
for i,j in zip(wordcloud['branch'],wordcloud['speaker']):
        participant_transcript.append(i)
        participants.append(j)

In [ ]:
# import pandas as pd
# import string
# import nltk
# from nltk.util import ngrams
# from gensim.parsing.preprocessing import remove_stopwords

# nlp = spacy.load("en_core_web_sm")

# for i,j in zip(participant_transcript, participants):
#     text = i
#     doc = nlp(text)
    
#     def extract_ngrams(data, num):
#         n_grams = ngrams(nltk.word_tokenize(data), num)
#         return [ ' '.join(grams) for grams in n_grams]


#     filtered_sentence = i.translate(str.maketrans('', '', string.punctuation))

# #     filtered_sentence = remove_stopwords(filtered_sentence)

#     bigrams_from_text = extract_ngrams(filtered_sentence, 2)
#     bigrams = pd.value_counts((bigrams_from_text)).head(3)
    
#     trigrams_from_text = extract_ngrams(filtered_sentence, 3)
#     trigrams = pd.value_counts((trigrams_from_text)).head(3)
    
#     quadgrams_from_text = extract_ngrams(filtered_sentence, 4)
#     quadgrams = pd.value_counts((quadgrams_from_text)).head(3)

#     print(F"PERSON {j}:")
#     print("2-word phrases")
#     print(bigrams)
#     print("\n")  

#     print("3-word phrases")
#     print(trigrams)
#     print("\n")  
    
#     print("4-word phrases")
#     print(quadgrams)
#     print("\n")  

In [ ]:
# 

In [ ]:
import pandas as pd
import string
import nltk
from nltk.util import ngrams
import spacy

nlp = spacy.load("en_core_web_sm")

# Example data (replace with your actual participant_transcript and participants)
# participant_transcript = [
#     "This is a sample text for participant 1.",
#     "Another text for participant 2 with some more words.",
# ]
# participants = ["Participant 1", "Participant 2"]

# Function to extract n-grams from text
def extract_ngrams(text, num):
    tokens = nltk.word_tokenize(text.lower())  # Tokenize and lowercase the text
    n_grams = ngrams(tokens, num)
    return [' '.join(grams) for grams in n_grams]

# Function to clean and filter text
def clean_text(text):
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Optionally remove stopwords (commented out for now)
    # text = remove_stopwords(text)
    return text

# Process each participant's transcript
for text, participant in zip(participant_transcript, participants):
    # Clean and preprocess text
    cleaned_text = clean_text(text)
    
    # Extract n-grams
    bigrams = pd.Series(extract_ngrams(cleaned_text, 2)).value_counts().head(3)
    trigrams = pd.Series(extract_ngrams(cleaned_text, 3)).value_counts().head(3)
    quadgrams = pd.Series(extract_ngrams(cleaned_text, 4)).value_counts().head(3)
    cincgrams = pd.Series(extract_ngrams(cleaned_text, 5)).value_counts().head(3)
    
    # Print results
    print(f"PERSON {participant}:")
    print("2-word phrases")
    print(bigrams)
    print("\n")
    print("3-word phrases")
    print(trigrams)
    print("\n")
    print("4-word phrases")
    print(quadgrams)
    print("\n")
    print("5-word phrases")
    print(cincgrams)
    print("\n")
    


### Open AI

In [ ]:
# !pip install openai

In [ ]:
# Never paste API keys into notebooks. Load them from .env instead.
import os
API_Key_OpenAI = os.getenv('OPENAI_API_KEY')


In [ ]:
# SocioLinguist = f"Act as an expert sociolinguist with a specialization in group facilitation & conversation analysis. What does it say about a person's conversational style if they use the followings phrases often in their speech? What indications of their personality or skills might be signaled by that verbal tic? Participant {participant} used the following 2-word phrase: {bigrams}, 3-word phrases: {trigrams}, 4-word phrases: {quadgrams}, 5-word phrases: {cincgrams}"

In [ ]:
# SocioLinguist

In [ ]:

# from openai import OpenAI
# client = OpenAI(api_key = API_Key_OpenAI)
# completion = client.chat.completions.create(
#     model="gpt-4o",
#     messages=[
#         {"role": "user", "content": SocioLinguist}
#     ]
# )


In [ ]:
# print(completion.choices[0].message.content)


### Prompt for each Participant

In [ ]:
for i in participants:
    print(i)

In [ ]:
max_interactions

In [ ]:
from IPython.display import Markdown

print(Markdown("**Hello, World!** This is *italic* and this is `code`."))

# PROMPT 001

## Function to clean and filter text
def clean_text(text):
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Optionally remove stopwords (commented out for now)
    # text = remove_stopwords(text)
    return text

# Process each participant's transcript
# Process each participant's transcript
for participant, transcript in zip(participants, participant_transcript):
    
    
    print(f"""Act as an expert in mastermind group facilitation. Generate a highly-actionable, thought-provoking report that analyzes a participant {participant}'s conversational style. 
    Use the data provided and all of the knowledge available to you when prescribing recommendations. 
    The report should be structured as follows:""")
    
    print(f"1. Metrics")  
    print(f"- Present key statistics in tabular format concisely and clearly.")
    print(f"- Analyze the follow metrics: Conversational Balance Score, Questions Asked, Speaking Time, Words per Minute, Interactions, Idea Generation, Unique Words (Idea Density). ")
    
    print(f"Conversational Balance Score:{equal_interaction_score}, where 100% is a perfectly equitable conversation.")
    print(f"{participant} asked {question_counts[participant]} of the {question_counts}questions.")

        # Extract n-grams
    cleaned_text = clean_text(text)
    bigrams = pd.Series(extract_ngrams(cleaned_text, 2)).value_counts().head(3)
    cincgrams = pd.Series(extract_ngrams(cleaned_text, 5)).value_counts().head(3)

    # Count the number of question marks as an indicator of questions asked
    num_questions = transcript.count("?")
    question_counts[participant] = num_questions
        
    print(f"Speaking time")
    percentage_time_spoken = talk_time_sorted
    group_high_time=max_percentage
    group_low_time=min_percentage
    group_avg_time=avg_percentage

    print(f"{percentage_time_spoken} of the time when the group high is {group_high_time}, the low is {group_low_time}, and the average is {group_avg_time}.")

    print(f"Words Per Minute")
    wpm=rate_of_speech
    wpm_high=max_rate
    wpm_low=min_rate
    wpm_avg=avg_rate
    
    print(f"{wpm} words per minute, while the group high is {wpm_high}, low is {wpm_low} and the average is {wpm_avg}.")
    print(f"Interactions between Participants")
    participant_most= interactions
    interactions_high= max_interactions
    interactions_low=min_interactions
    # interactions_avg=interactions_avg
    print(f" {participant_most}: The most interactions {interactions_high}, the fewest {interactions_low}.")


    print(f"Idea Generation")
    ideas=idea_counts,
    ideas_high=max_ideas,
    ideas_low=min_ideas,
    ideas_avg=group_average,
    
    print(f"Participant {participant} total number of ideas generated in the conversation are {ideas} while the group high is {ideas_high}, the low is {ideas_low}, and the average is {ideas_avg}")

    print(f"Unique Words")
    unique_words=unique_word_percentages
    print(f"Use participant {participant}'s unique words to calculate 'idea density': {unique_words}.")

    # unique_high = unique_word_percentages,
    # unique_low = unique_word_percentages,
    # unique_avg = unique_word_percentages,

    print(f"Questions")
    questions=question_counts,
    questions_high=max_questions,
    questions_low=min_questions,
    questions_avg=average_questions,
    print(f"Participant {participant} total number of questions generated in the conversation are {questions} where the group high is {questions_high} the low is {questions_low} and the average is {questions_avg}.")    

    def extract_ngrams(text, num):
        tokens = nltk.word_tokenize(text.lower())  # Tokenize and lowercase the text
        n_grams = ngrams(tokens, num)
        return [' '.join(grams) for grams in n_grams]

    cleaned_text = clean_text(text)

    print(f"Extract bigrams & other phrases that would help participant {participant} communicate more effectively.")
    print('\n')

    print(f"2. What It All Means  - {participant}’s conversational style with a peer-group mastermind: Identify key positive behaviors & share their impact on group engagement.  Extract two verbatim quotes from the provided transcript that strengthen your argument about {participant}’s positive contributions. Here's the transcript: {transcript}")
    print('\n')
    print(f"3. Application  - Provide specific recommendations for {participant} to improve persuasiveness & executive presence.")

    print(f"Analyze participant {participant}'s speech in terms of CONVERGENT & DIVERGENT thinking.")
    # print(f"Suggest new sentence starters, phrases, and stems that would strengthen the bond of the group. Ensure the recommendations are directly applicable in professional settings.")
    print(f"The goal is to deliver a clear prescriptive analysis that builds participant {participant}’s self-awareness.")
    
    print('\n')

# PROMPT 002

In [ ]:
# Function to clean and filter text
def clean_text(text):
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Optionally remove stopwords (commented out for now)
    # text = remove_stopwords(text)
    return text

# Process each participant's transcript
# Process each participant's transcript
for participant, transcript in zip(participants, participant_transcript):
    
    
    print(f"""Instructions: Analyze the conversation and extract the requested data. Present all metrics in a single consolidated table without additional explanation.""")
    
    print(f"1. Metrics")  
    print(f"- Clearly and concisely, create a a table that presents the following metrics: Conversational Balance Score, Questions Asked, Speaking Time, Words per Minute, Interactions, Idea Generation, Unique Words (Idea Density). ")
    
    print(f"Conversational Balance Score:{equal_interaction_score}, where 100% is a perfectly equitable conversation.")
    print(f"{participant} asked {question_counts[participant]} of the {question_counts}questions.")

        # Extract n-grams
    cleaned_text = clean_text(text)
    bigrams = pd.Series(extract_ngrams(cleaned_text, 2)).value_counts().head(3)
    cincgrams = pd.Series(extract_ngrams(cleaned_text, 5)).value_counts().head(3)

    # Count the number of question marks as an indicator of questions asked
    num_questions = transcript.count("?")
    question_counts[participant] = num_questions
        
    print(f"Speaking time")
    percentage_time_spoken = talk_time_sorted
    group_high_time=max_percentage
    group_low_time=min_percentage
    group_avg_time=avg_percentage

    print(f"{percentage_time_spoken} of the time when the group high is {group_high_time}, the low is {group_low_time}, and the average is {group_avg_time}.")

    print(f"Words Per Minute")
    wpm=rate_of_speech
    wpm_high=max_rate
    wpm_low=min_rate
    wpm_avg=avg_rate
    
    print(f"{wpm} words per minute, while the group high is {wpm_high}, low is {wpm_low} and the average is {wpm_avg}.")
    print(f"Interactions between Participants")
    participant_most= interactions
    interactions_high= max_interactions
    interactions_low=min_interactions
    # interactions_avg=interactions_avg
    print(f" {participant_most}: The most interactions {interactions_high}, the fewest {interactions_low}.")


    print(f"Idea Generation")
    ideas=idea_counts,
    ideas_high=max_ideas,
    ideas_low=min_ideas,
    ideas_avg=group_average,
    
    print(f"Participant {participant} total number of ideas generated in the conversation are {ideas} while the group high is {ideas_high}, the low is {ideas_low}, and the average is {ideas_avg}")

    print(f"Unique Words")
    unique_words=unique_word_percentages
    print(f"Use participant {participant}'s unique words to calculate 'idea density': {unique_words}.")

    # unique_high = unique_word_percentages,
    # unique_low = unique_word_percentages,
    # unique_avg = unique_word_percentages,

    print(f"Questions")
    questions=question_counts,
    questions_high=max_questions,
    questions_low=min_questions,
    questions_avg=average_questions,
    print(f"Participant {participant} total number of questions generated in the conversation are {questions} where the group high is {questions_high} the low is {questions_low} and the average is {questions_avg}.")    

    def extract_ngrams(text, num):
        tokens = nltk.word_tokenize(text.lower())  # Tokenize and lowercase the text
        n_grams = ngrams(tokens, num)
        return [' '.join(grams) for grams in n_grams]

    cleaned_text = clean_text(text)
    
    
    print('\n')
    print(f"2. Key Contributions - Extract three verbatim quotes, lightly edited for legibility, from participant {participant} in the transcript that highlight their contribution to the discussion. Here's the transcript: {transcript}. No need for in-depth analysis or additional commentary.")
    print('\n')
    print(f"3. Next Steps")
    print(f'- Extract and paraphrase all **explicit and implicit** action items that **Participant {participant}** has agreed to complete.  - Present the action items as a **plain bullet-point list** without additional commentary.  - Keep each action item **short and direct**, ensuring clarity and actionability.  - Maintain consistency with the autogenerated report format, as this is the final section.  ')

    print('\n')

In [ ]:
trigrams

In [ ]:
# SocioLinguist

In [ ]:
# Conversational_Health_Report = """
# **Prompt:**  

# Act as a Harvard Business School professor specializing in sociolinguistics, group facilitation, and conversation analysis, with additional expertise as a Data Scientist** skilled in **Mathematics and Natural Language Processing.  

# **Task:**  
# Generate a **brief, highly actionable, executive-level report** analyzing **Participant {participant}'s long-term conversational style**, using **data-driven insights** and **prescriptive recommendations**. The report should be structured as follows:  

# #### **1. Metrics** (in tabular format with short story-based summary)  
# - Present **key statistics** concisely and clearly.  
# - Emphasize **Conversational Balance Score:{equal_interaction_score}, speaking time, words per minute, interactions, idea generation, unique words, and question-asking behavior**.  
# - Participant {participant} speaks {time_share} of the time when the group high is {group_high_time}, the low is {group_low_time}, and the average is {group_avg_time}.
# Participant {participant} speaks {wpm} words per minute when the group high is {wpm_high}, the low is {wpm_low}, and the average is {wpm_avg}.
# Participant {participant} total interactions are {participant_most} while the group high is {interactions_high}, the low is {interactions_low}, and the average is {interactions_avg}.
# Participant {participant} total number of ideas generated in the conversation are {ideas} while the group high is {ideas_high}, the low is {ideas_low}, and the average is {ideas_avg}.
# Participant {participant} unique words are part of the following distribution {unique_words}.
# Participant {participant} total number of questions generated in the conversation are {questions} while the group high is {questions_high}, the low is {questions_low}, and the average is {questions_avg}.
# - Where possible, **highlight outliers, trends, and potential impact** on group dynamics.  

# #### **2. What It All Means**  
# - Use the **Character, Goal, Challenge, and Resolution** framework to narrate **Participant {participant}’s long-term conversational tendencies**.  
# - Identify **key behavioral patterns** and their **implications for group engagement, leadership presence, and influence**.  
# - Extract **at least two verbatim quotes** from the provided transcript to **illustrate {participant}’s conversational habits**. Here's the transcript: {transcript}

# #### **3. Application**  
# - Provide **specific, behavioral recommendations** for {participant} to improve **persuasiveness, engagement, and conversational balance**.  
# - Based on **bigrams, trigrams, identify:  
#   - **Problematic sentence structures** (if any) and their potential impact.  
#   - **New sentence starters, phrases, and stems** to enhance **persuasive communication**.  
# - Ensure the recommendations are **directly applicable in professional settings**.
# - {bigrams}
# - {cincgrams}

# The **goal** is to deliver a **clear, data-backed, and prescriptive analysis** that **enhances {participant}’s effectiveness as a communicator in group discussions**.  

# """

# Conversational_Health_Report = Conversational_Health_Report.format(
#     equal_interaction_score = equal_interaction_score,
#     transcript = participant_transcript[0],
#     participant=participant,
#     time_share=talk_time_sorted,
#     group_high_time=max_percentage,
#     group_low_time=min_percentage,
#     group_avg_time=avg_percentage,
#     wpm=rate_of_speech,
#     wpm_high=max_rate,
#     wpm_low=min_rate,
#     wpm_avg=avg_rate,
#     participant_most= interactions,
#     interactions_high= max_interactions,
#     interactions_low=min_interactions,
#     interactions_avg=group_average,
#     ideas=idea_counts,
#     ideas_high=max_ideas,
#     ideas_low=min_ideas,
#     ideas_avg=group_average,
#     unique_words=unique_word_percentages,
#     unique_high = unique_word_percentages,
#     unique_low = unique_word_percentages,
#     unique_avg = unique_word_percentages,
#     questions=question_counts,
#     questions_high=max_questions,
#     questions_low=min_questions,
#     questions_avg=average_questions,
#     bigrams=bigrams,
#     trigrams=trigrams,
#     quadgrams=quadgrams,
#     cincgrams=cincgrams
# )

# print(Conversational_Health_Report)

In [ ]:
# bigrams

In [ ]:
# SocioLinguist = """Act as a Harvard Business School professor with a speciality in sociolinguistics, group facilitation & conversation analysis. 
# You also have an additional skillset as an expert Data Scientist with a background in Mathematics and Natural Language Processing. 

# Generate a behavior-based report for discussion participant {participant} based on the following statistics. Make it clear and concise, but packed with value.
# Use headings "Metrics", "What it all Means", and "Application"

# 1. "Metrics"
# Overall, the Conversational Balance Score amongst all participants was {equal_interaction_score}.

# Participant {participant} speaks {time_share} of the time when the group high is {group_high_time}, the low is {group_low_time}, and the average is {group_avg_time}.

# Participant {participant} speaks {wpm} words per minute when the group high is {wpm_high}, the low is {wpm_low}, and the average is {wpm_avg}.

# Participant {participant} total interactions are {participant_most} while the group high is {interactions_high}, the low is {interactions_low}, and the average is {interactions_avg}.

# Participant {participant} total number of ideas generated in the conversation are {ideas} while the group high is {ideas_high}, the low is {ideas_low}, and the average is {ideas_avg}.

# Participant {participant} unique words are {unique_words} while the group high is {unique_high}, the low is {unique_low}, and the average is {unique_avg}.

# Participant {participant} total number of questions generated in the conversation are {questions} while the group high is {questions_high}, the low is {questions_low}, and the average is {questions_avg}.

# 2. "What it all Means"
# Using the storytelling framework of "Character, Goal, Challenge, and Resolution", write a short report that summarizes the conversational style of participant {participant}. Pull at least 2 verbatim quotes from the following transcript {transcript}.

# 3. "Application"

# Act as the same expert in persuasion & linguistics described above. Based on the bigrams, trigrams, quadgrams, and 5-word phrases , what are “sentence starters, phrases, and stems” that could help participant {participant} be more persuasive?
# {bigrams} trigams {trigrams}, quadgrams {quadgrams}, and 5-word phrase {cincgrams}


# """




# SocioLinguist = SocioLinguist.format(
#     equal_interaction_score = equal_interaction_score,
#     transcript = participant_transcript[0],
#     participant=participant,
#     time_share=max_percentage,
#     group_high_time=max_percentage,
#     group_low_time=min_percentage,
#     group_avg_time=avg_percentage,
#     wpm=rate_of_speech,
#     wpm_high=max_rate,
#     wpm_low=min_rate,
#     wpm_avg=avg_rate,
#     participant_most= max_interactions,
#     interactions_high= max_interactions,
#     interactions_low=min_interactions,
#     interactions_avg=group_average,
#     ideas=total_ideas,
#     ideas_high=max_ideas,
#     ideas_low=min_ideas,
#     ideas_avg=group_average,
#     unique_words=unique_words_count,
#     unique_high = percentage_unique,
#     unique_low = percentage_unique,
#     unique_avg = percentage_unique,
#     questions=question_counts,
#     questions_high=max_questions,
#     questions_low=min_questions,
#     questions_avg=average_questions,
#     bigrams=bigrams,
#     trigrams=trigrams,
#     quadgrams=quadgrams,
#     cincgrams=cincgrams
# )

# print(SocioLinguist)


# DONUTS Bi Grams

In [ ]:
# import pandas as pd
# import string
# import nltk
# from nltk.util import ngrams
# import matplotlib.pyplot as plt

# # Ensure you have the required NLTK data
# nltk.download('punkt')

# # Sample data (replace with your actual participant_transcript and participants)
# # participant_transcript = [
# #     "This is a sample text for participant 1.",
# #     "Another text for participant 2 with some more words."
# # ]
# # participants = ["Participant 1", "Participant 2"]

# # Function to extract n-grams from text
# def extract_ngrams(text, num):
#     tokens = nltk.word_tokenize(text.lower())  # Tokenize and lowercase the text
#     n_grams = ngrams(tokens, num)
#     return [' '.join(grams) for grams in n_grams]

# # Function to clean and filter text
# def clean_text(text):
#     # Remove punctuation
#     text = text.translate(str.maketrans('', '', string.punctuation))
#     return text

# # Process each participant's transcript
# for text, participant in zip(participant_transcript, participants):
#     # Clean and preprocess text
#     cleaned_text = clean_text(text)
    
#     # Extract bigrams
#     bigrams = pd.Series(extract_ngrams(cleaned_text, 2)).value_counts().head(3)
    
#     # Plot donut chart
#     plt.figure(figsize=(8, 8))
#     plt.pie(bigrams, labels=bigrams.index, autopct='%1.1f%%', startangle=140, wedgeprops=dict(width=0.3))
#     plt.title(f"Top 3 Bigrams for {participant}")
#     # plt.show()


# Dimensionality

In [ ]:
# import matplotlib.pyplot as plt
# import networkx as nx
# from nltk.util import ngrams
# from nltk.tokenize import word_tokenize
# from gensim.parsing.preprocessing import remove_stopwords
# import string
# import nltk

# # Ensure you have the required NLTK data
# nltk.download('punkt')

# # Sample participant transcripts (replace with your actual data)
# # participant_transcript = [
# #     "I like to play soccer with my friends.",
# #     "Soccer is my favorite sport to watch on TV.",
# #     "My friends and I often play board games after school."
# # ]
# # participants = ["A", "B", "C"]

# # Function to extract bigrams
# def extract_bigrams(data):
#     tokens = word_tokenize(data.lower())
#     bigrams = ngrams(tokens, 2)
#     return [' '.join(gram) for gram in bigrams]

# # Function to preprocess text (remove punctuation and stopwords)
# def preprocess_text(text):
#     text = text.translate(str.maketrans('', '', string.punctuation))
#     text = remove_stopwords(text)
#     return text

# # Process each participant's transcript
# for text, speaker in zip(participant_transcript, participants):
#     text = preprocess_text(text)
#     bigrams_from_text = extract_bigrams(text)
    
#     # Interactive Network Visualization for Bigrams
#     G = nx.Graph()
#     for bigram in bigrams_from_text:
#         words = bigram.split()
#         if len(words) == 2:
#             G.add_edge(words[0], words[1])
    
#     pos = nx.spring_layout(G)
#     plt.figure(figsize=(50, 30))
#     nx.draw(G, pos, with_labels=True, node_color='skyblue', node_size=1500, edge_color='grey', linewidths=1, font_size=12)
#     plt.title(f"Interactive Network Visualization - Participant {speaker}")
#     plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from nltk.util import ngrams
from nltk.tokenize import word_tokenize
from collections import Counter
from gensim.parsing.preprocessing import remove_stopwords
import string
import nltk

# Ensure you have the required NLTK data
nltk.download('punkt')

# Sample participant transcripts (replace with your actual data)
# participant_transcript = [
#     "I like to play soccer with my friends.",
#     "Soccer is my favorite sport to watch on TV.",
#     "My friends and I often play board games after school."
# ]
# participants = ["A", "B", "C"]

# Function to extract bigrams
def extract_bigrams(data):
    tokens = word_tokenize(data.lower())
    bigrams = ngrams(tokens, 3)
    return [' '.join(gram) for gram in bigrams]

# Function to preprocess text (remove punctuation and stopwords)
def preprocess_text(text):
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = remove_stopwords(text)
    return text

# Aggregate all bigrams across all participants
all_bigrams = []

# Process each participant's transcript
for text, speaker in zip(participant_transcript, participants):
    text = preprocess_text(text)
    bigrams_from_text = extract_bigrams(text)
    all_bigrams.extend(bigrams_from_text)

# Count the most common bigrams
bigram_counts = Counter(all_bigrams)
common_bigrams = bigram_counts.most_common(10)  # Adjust the number to show more/less bigrams

# Prepare data for bar graph
bigrams = [bigram for bigram, count in common_bigrams]
counts = [count for bigram, count in common_bigrams]

# Let's assume you have N speakers
N = len(rate_of_speech['speaker'])
colors = plt.cm.viridis(np.linspace(0, 1, N))  # Generates an array of colors from the viridis colormap


# Plotting the bar graph
plt.figure(figsize=(12, 6))
plt.barh(bigrams, counts, color=colors)
plt.xlabel('Frequency')
plt.ylabel('Trigrams')
plt.title('Most Common Trigrams')
plt.gca().invert_yaxis()  # Invert y-axis to show the highest frequency on top
plt.show()


# PCA Component 

In [ ]:
# import pandas as pd
# import string
# import nltk
# from nltk.util import ngrams
# from nltk.tokenize import word_tokenize
# from gensim.parsing.preprocessing import remove_stopwords
# import matplotlib.pyplot as plt
# import networkx as nx
# from sklearn.feature_extraction.text import CountVectorizer
# from sklearn.decomposition import PCA
# import spacy

# # Ensure you have the required NLTK data
# nltk.download('punkt')

# # Initialize spaCy
# nlp = spacy.load("en_core_web_sm")

# # Sample participant transcripts (replace with your actual data)
# # participant_transcript = [
# #     "I like to play soccer with my friends.",
# #     "Soccer is my favorite sport to watch on TV.",
# #     "My friends and I often play board games after school."
# # ]
# # participants = ["A", "B", "C"]

# # Function to extract n-grams
# def extract_ngrams(data, num):
#     tokens = word_tokenize(data.lower())
#     n_grams = ngrams(tokens, num)
#     return [' '.join(grams) for grams in n_grams]

# # Function to preprocess text (remove punctuation and stopwords)
# def preprocess_text(text):
#     text = text.translate(str.maketrans('', '', string.punctuation))
#     text = remove_stopwords(text)
#     return text

# # Process each participant's transcript
# for text, speaker in zip(participant_transcript, participants):
#     text = preprocess_text(text)
#     bigrams_from_text = extract_ngrams(text, 2)
#     trigrams_from_text = extract_ngrams(text, 3)
#     quadgrams_from_text = extract_ngrams(text, 4)
    
#     # Option 2: Interactive Network Visualization
# #     G = nx.Graph()
# #     for ngram in bigrams_from_text + trigrams_from_text + quadgrams_from_text:
# #         words = ngram.split()
# #         for i in range(len(words)-1):
# #             G.add_edge(words[i], words[i+1])
    
# #     pos = nx.spring_layout(G)
# #     plt.figure(figsize=(10, 6))
# #     nx.draw(G, pos, with_labels=True, node_color='skyblue', node_size=1500, edge_color='grey', linewidths=1, font_size=12)
# #     plt.title(f"Interactive Network Visualization - Participant {speaker}")
# #     plt.show()
    
#     # Option 3: Dimensionality Reduction and Scatter Plot
#     all_ngrams = bigrams_from_text + trigrams_from_text + quadgrams_from_text
#     vectorizer = CountVectorizer(ngram_range=(2, 4))
#     X = vectorizer.fit_transform(all_ngrams)
#     X_reduced = PCA(n_components=2).fit_transform(X.toarray())
    
#     plt.figure(figsize=(30, 18))
#     plt.scatter(X_reduced[:, 0], X_reduced[:, 1], c='b', marker='o', alpha=0.5)
#     for i, txt in enumerate(all_ngrams):
#         plt.annotate(txt, (X_reduced[i, 0], X_reduced[i, 1]))
#     plt.title(f"Dimensionality Reduction and Scatter Plot - Participant {speaker}")
#     plt.xlabel("PCA Component 1")
#     plt.ylabel("PCA Component 2")
#     plt.grid(True)
#     # plt.show()


In [ ]:
# # import nltk

# #WP gives more ideas; NN is nouns
# # modal auxiliary
# for i, j in zip(participant_transcript, participants):
#     is_noun = lambda pos: pos[:2] == 'MD'

#     # do the nlp stuff
#     tokenized = nltk.word_tokenize(i)
#     nouns = [word for (word, pos) in nltk.pos_tag(tokenized) if is_noun(pos)] 

#     from collections import Counter
 
#     # creating a list with the keys
#     items = Counter(nouns).keys()
#     num_ideas = len(items)
#     print(f"Person {j} generated {num_ideas} ideas.")
    
#     # {items}

# Ums, Ahs, Uh per Person

In [ ]:
import matplotlib.pyplot as plt

# Sample participant transcripts and participants

# Initialize a dictionary to store counts of filler words
filler_counts = {speaker: {' uh': 0, ' um': 0, ' ah': 0} for speaker in participants}

# Process each participant's transcript
for text, speaker in zip(participant_transcript, participants):
    fillers_uh = text.lower().count(" uh")
    fillers_ah = text.lower().count(" ah")
    fillers_um = text.lower().count(" um")
    
    filler_counts[speaker]['uh'] = fillers_uh
    filler_counts[speaker]['ah'] = fillers_ah
    filler_counts[speaker]['um'] = fillers_um
    
    total_fillers = fillers_uh + fillers_ah + fillers_um
    print(f"Person {speaker} had {total_fillers} fillers. \n{fillers_uh} - Uh: expresses uncertainty; possible lack of safety\n{fillers_ah} - Ah: \n{fillers_um} - um")
    print()

# Prepare data for stacked bar graph
speakers = list(filler_counts.keys())
uh_counts = [filler_counts[speaker]['uh'] for speaker in speakers]
um_counts = [filler_counts[speaker]['um'] for speaker in speakers]
ah_counts = [filler_counts[speaker]['ah'] for speaker in speakers]

# Plotting the stacked bar graph
bar_width = 0.5
index = range(len(speakers))

plt.figure(figsize=(10, 6))
plt.bar(index, uh_counts, bar_width, label='Uh', color='skyblue')
plt.bar(index, um_counts, bar_width, bottom=uh_counts, label='Um', color='lightgreen')
plt.bar(index, ah_counts, bar_width, bottom=[i+j for i,j in zip(uh_counts, um_counts)], label='Ah', color='lightcoral')

plt.xlabel('Participants')
plt.ylabel('Count of Filler Words')
plt.title('Count of "Uh", "Um", and "Ah" by Each Participant')
plt.xticks(index, speakers)
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
#novel noun intro = 
#new idea = topic modeling then new cluster in topic modeling
#modal into new nouns = that unit --> idea


<!-- ***Simplifying Text*** -->

In [ ]:
# research_paper = """
# As a teenager, it's normal to feel stressed out from time to time. Whether it's due to academic pressure, social anxieties, or personal development issues, stress is a common part of adolescence. 

# However, if not properly managed, stress can have negative effects on your physical and mental health. Luckily, there are a number of coping skills and stress management techniques that can help. things like hot yoga, drinking plenty of water, meditation, and talk therapy have all been proven to be effective in managing stress. So if you're feeling overwhelmed, don't hesitate to reach out for help and explore different coping mechanisms until you find what works best for you. Coping with stress is an important part of growing up, and by learning how to manage it effectively now, you'll be setting yourself up for success in the future.
# """

In [ ]:
# listRes = list(research_paper.split(" ."))

In [ ]:
# listRes

In [ ]:
# #Loop through the list and determine which sentence is most important
# for i in listRes:
#     article_parser = PlaintextParser.from_string(i, Tokenizer(LANGUAGE))
#     article_stemmer = Stemmer(LANGUAGE)

#     article_summarizer = Summarizer(article_stemmer)
#     article_summarizer.stop_words = get_stop_words(LANGUAGE)
#     article_sentence = summarizer(article_parser.document, SENTENCES_COUNT)

#     print(f"{article_sentence[0]}")
#     print("\n")

In [ ]:
# words_person = words_person.drop(['UNK'])

words_person = df[['speaker', 'num_words_spoken']]
# words_person.drop(['UNK']).index

words_person = words_person.groupby('speaker').sum('num_words_spoken')

words_person['speaker'] = words_person.index
drop_unknown = words_person[words_person['speaker'] == 'UNK'].index

words_person.drop(drop_unknown, inplace=True)

# Individualized Feedback
### How much did each person speak, compared to their peers?

In [ ]:
import math 
for i,j in zip(words_person['speaker'], words_person['num_words_spoken']):

    times_each_spoke = pd.crosstab(df['speaker'], columns = 'speaker')
    times_each_spoke = times_each_spoke['speaker'].sort_values(ascending = False)
    
    average_num_words_spoken = words_person['num_words_spoken'].mean()
    average_num_words_spoken

    conversation_total = words_person['num_words_spoken'].sum()
    
    participant_words = words_person['num_words_spoken'][i]
    participant_words

    pct_change = (participant_words - average_num_words_spoken)/average_num_words_spoken
    pct_change = pct_change*100
    pct_change = pct_change.round()
    
    pct_of_conversation = (j/conversation_total)*100

    sign = []
    if pct_change > 0:
        sign.append('more')
    if pct_change < 0:
        sign.append('less')
        
    print(f"Person {i}, you spoke {participant_words} words.")
    print(f"That was {pct_of_conversation.round()}% of the conversation.")
    print(f"You spoke {times_each_spoke[i]} times for a total of {(j)/650} minutes. ")
    print(f"You spoke {np.abs(pct_change)}% {sign[0].upper()} than your peers, who - on average - spoke {math.floor(average_num_words_spoken)} words.")
    print('\n')

### LIWC

In [ ]:
# !pip install liwc

# Plotly graph of 'Who spoke the most'

In [ ]:
# df.head()
plotly = df[['speaker', 'num_words_spoken']]
plotly = plotly.groupby('speaker').sum('num')
# plotly = plotly.drop(['UNK'])

bubble = plotly['num_words_spoken'].sort_values(ascending = False)
bubble = bubble.to_frame()
bubble['speaker'] = bubble.index

import plotly.express as px

fig = px.scatter(bubble, x="num_words_spoken", y="speaker",
	         size="num_words_spoken", color="speaker",
                 hover_name="speaker", log_x=True, size_max=60)
fig.write_html('harkness_bubble.html')

***How many words are unique to this conversation?***

In [ ]:
final_transcript = []
for i,j in zip(df['text'], df['speaker']):
    final_transcript.append(i)
    
listToStr = ' '.join(final_transcript)
res = re.sub(r'[^\w\s]', '', listToStr)
 
# storing string in the form of list of words
s = set(res.split(" "))


In [ ]:
word_count = res.split(" ")
word_count = len(word_count)
pct_unique_words = (len(s)/word_count)*100

In [ ]:
print(f"{round(pct_unique_words)}% of the {word_count} words in this conversation were unique.")

***How many unique words/person***

In [ ]:
for i, j in zip(participant_transcript, participants):
    unique = set(i.split(" "))
    unique = len(unique)
    participant_words = len(i.split(" "))
    print(f"{unique} of Person {j}'s {participant_words} words were unique to them. That's {(unique/participant_words)*100}")

### Concordance

<!-- ### <span style="color:#17499e"><b>Concordance: </span><span style="color:#2FC2F2"> Where & when words are used -->

In [ ]:
# # # # (Un)comment the code below, if you don't want to waste assembly.ai credits
df['num_words_spoken'] = df['text'].str.split(" ")
df['num_words_spoken'] = df['num_words_spoken'].str.len()

In [ ]:
audio= []
for i in df['text']:
    audio.append(i)

In [ ]:
conversation_audio  = "".join(audio)

In [ ]:
# Uses stopwords for english from NLTK, and all puntuation characters by
# default
from rake_nltk import Rake
r = Rake()

# Extraction given the text.
r.extract_keywords_from_text(conversation_audio)

# To get keyword phrases ranked highest to lowest.
r.get_ranked_phrases()

# To get keyword phrases ranked highest to lowest with scores.
r.get_ranked_phrases_with_scores()

for rating, keyword in r.get_ranked_phrases_with_scores():
    if rating > 15:
        print(rating, keyword)

In [ ]:
r.extract_keywords_from_text(conversation_audio)

In [ ]:
# r.get_ranked_phrases_with_scores()

In [ ]:
import nltk
from nltk.corpus import stopwords

In [ ]:
# filtered_sentence = []

# stop_words = set(stopwords.words('english'))

# for w in text_KEE:
#     if w not in stop_words:
#         filtered_sentence.append(w)

In [ ]:
FindWordsText_KEE = nltk.word_tokenize(conversation_audio)

In [ ]:
text_KEE = nltk.Text(FindWordsText_KEE)

In [ ]:
word_of_interest = 'sales'

In [ ]:
multiple_word = ['goals', 'trend', "our"]

In [ ]:
most_common_words = ['you']

In [ ]:
print(f"'{word_of_interest}' appeared in the conversation {text_KEE.count(word_of_interest)} times.")

In [ ]:
# r.get_ranked_phrases_with_scores()

In [ ]:
# #10 Most common words
# from nltk.probability import FreqDist
# fdist = FreqDist(filtered_sentence)

# #print the 10 most common words
# most_common_words = fdist.most_common(10)
# most_common_words

In [ ]:
#You can look at each word in context and get a sense of how it's used in the conversation.
text_KEE.concordance(word_of_interest)
# text_KEE.concordance(most_common_words)

In [ ]:
text_KEE.similar(word_of_interest)

In [ ]:
# unigrams = list(unigrams.index)

In [ ]:
# unigrams.index

In [ ]:
# most_common_words

In [ ]:
#When does a CERTAIN word get used in the conversation
text_KEE.dispersion_plot([word_of_interest])

#When does a GROUP of words get used in the conversation
text_KEE.dispersion_plot(multiple_word)

#When did the top 10 words occur in the conversation?
text_KEE.dispersion_plot(most_common_words)

#Unigrams
# text_KEE.dispersion_plot(unigrams)

# <span style="color:#17499e"><b>Verbatim </span><span style="color:#2FC2F2"> Transcripts & Other Topics

In [ ]:
final_transcript = []
for i,j in zip(df['text'], df['speaker']):
    final_transcript.append(i)
    print(f"Person {j} - {i}")
    print("\n")

# <span style="color:#17499e"><b>EMOTIONAL DATA</span><span style="color:#2FC2F2">arch

In [ ]:
# Read JSON file into a Pandas DataFrame
DATA = pd.read_json('HUME_007.json')

# Display the DataFrame
print(DATA)

In [ ]:
# Loop through each text and associated emotions
for prediction in DATA['results'][0]['predictions']:
    for pred in prediction['models']['prosody']['grouped_predictions']:
        for text_emotion in pred['predictions']:
            # Extract text and emotions
            text = text_emotion['text']
            emotions = text_emotion['emotions']
            
            # Sort emotions by score in descending order
            sorted_emotions = sorted(emotions, key=lambda x: x['score'], reverse=True)
            
            # Print text
            print(f'{text}')
            print()
            
            # Print top 5 emotions and their scores
            # print('Top 5 Emotions:')
            for i, emotion in enumerate(sorted_emotions[:5], start=1):
                # print(f"{i}. '{emotion['name']}' - {emotion['score']}")
                print(f"{emotion['name']} - {emotion['score']*100}%")
            print()

In [ ]:
# sentiment.groupby(['sentiment'])['speaker'].value_counts()

***Important Names, Place, Dates***

In [ ]:
# entities = pd.DataFrame.from_dict(transcript_json['entities'])
# entities = entities[['entity_type', 'text']]

# entities.to_csv('entities.csv')
# entities.head()

In [ ]:
# entities.groupby('entity_type').describe()

In [ ]:
#This is VALUABLE
# entities.groupby('entity_type')['text'].value_counts().head(50)

In [ ]:
# entities.groupby('text')['entity_type'].value_counts().head(50)

### 3. Rate of speech

In [ ]:
# # # # #Uncomment the code below, if you're using the assembly.ai API.
# def make_transcript_from_json():
#     df = pd.DataFrame.from_dict(transcript_json['utterances'])
#     df['num_words_spoken'] = df['text'].str.len()
# df.to_csv('JBClass005.csv')
#     return df.head()

In [ ]:
# make_transcript_from_json()

In [ ]:
# # # # # (Un)comment the code below, if you don't want to waste assembly.ai credits
# df = pd.read_csv('0510_WeeklyTactical.csv')

# df['num_words_spoken'] = df['text'].str.split(" ")
# df['num_words_spoken'] = df['num_words_spoken'].str.len()

# INFO LEVEL: Organizational

1. Transcript,
2. Trending words,
3. Keywords/phrases over time.

# 3. Transcript

# What topics are trending?

In [ ]:
auto_highlights = pd.DataFrame.from_dict(transcript_json['auto_highlights_result'])

auto_highlights = auto_highlights['results']
auto_highlights.to_csv('SChighlights.csv')

# auto_highlights = pd.read_csv('highlights.csv')
auto_highlights

# trending = []
for i in auto_highlights:
    print(f"{i['text']} - {i['count']} Mentions")
    # print("\n")

#     trending.append(i['count'])
#     trending.append(i['text'])

# Google Search based Key terms

In [ ]:
key_words = []
for i in auto_highlights:
    key_words.append(i['text'])

In [ ]:
key_words
search_words = ' '.join(key_words)

In [ ]:
search_words

In [ ]:
key_words

In [ ]:
# from googlesearch import search

# print('3 suggested articles based on conversation keywords') 
# for j in search(search_words, num_results=2):
#     print(j)

***Topics***

In [ ]:
topics = transcript_json['iab_categories_result']['summary']
topics = pd.DataFrame.from_dict(topics, orient = 'index')
topics = topics[0].sort_values(ascending = False)
topics.to_csv('SCtopics.csv')
# topics.read_csv('topics.csv')
topics.head()

### Exact Quotes

In [ ]:
df['branch'] = df.groupby(['speaker'])['text'].transform(lambda x : ' '.join(x))
wordcloud = df[['speaker', 'branch']].drop_duplicates()
UNK = wordcloud[wordcloud['speaker'] == 'UNK'].index
wordcloud.drop(UNK, inplace= True)


participants = []
participant_transcript = []
for i,j in zip(wordcloud['branch'],wordcloud['speaker']):
        participant_transcript.append(i)
        participants.append(j)

In [ ]:
%timeit
from sumy.parsers.html import HtmlParser
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer as Summarizer
from sumy.nlp.stemmers import Stemmer
from sumy.utils import get_stop_words

In [ ]:
LANGUAGE = "english"
SENTENCES_COUNT = 3

In [ ]:
for i,j in zip(participant_transcript, participants):
    parser = PlaintextParser.from_string(i, Tokenizer(LANGUAGE))
    stemmer = Stemmer(LANGUAGE)

    summarizer = Summarizer(stemmer)
    summarizer.stop_words = get_stop_words(LANGUAGE)
    sentence = summarizer(parser.document, SENTENCES_COUNT)
    
    print(f"HEADLINE: Person {j}")
    print(f"'{sentence[0]}'")
    print("\n")

In [ ]:
sentiment.groupby('speaker').sentiment.nunique()

In [ ]:
sentiment.groupby('speaker').max()

In [ ]:
sentiment.groupby('speaker').describe()

In [ ]:
sentiment.groupby('speaker')

In [ ]:
# sentiment = sentiment[['speaker', 'sentiment']].drop_duplicates()
# sentiment.sortby('speaker')

In [ ]:
sentiment.index

In [ ]:
# for i,j in zip(sentiment['speaker'], sentiment['sentiment']):
#         print(f"Person {i}:")
#         print(j)

In [ ]:
# for i, j in zip(sentiment['speaker'], sentiment['sentiment']):
#     print()

How positive negative neutral is each person?

In [ ]:
sentiment

In [ ]:
sentiment['speaker'].value_counts()

In [ ]:
# How positive negative neutral is each person

***Sentiment Analysis***

In [ ]:
# for i in transcript_json['sentiment_analysis_results']:
# #     print(i['text'])
# #     print(i['sentiment'])
#     print(i['speaker'])
# #     print(i)
#     print(i['sentiment'])
    

***Entity: Person, Occupation, Organization, Location?***

In [ ]:
# for i in transcript_json['entities']:
#     print(i['entity_type'])
#     print(i['text'])
#     print('\n')

#     what_type = i['entity_type']
#     print(what_type)

In [ ]:
# df = pd.DataFrame.from_dict(transcript_json['utterances'])

# df['num_words_spoken'] = df['text'].str.split(" ")
# df['num_words_spoken'] = df['num_words_spoken'].str.len()
# df['num_words_spoken']

# df.to_csv('HUME_003.csv')

In [ ]:
# df

In [ ]:
# response.json()['id']
# '5emurg861-e855-4a21-b092-c2dd3149f153'

# INFO LEVEL: Managerial

1. Word Count Visualization
2. Chord Diagram
3. Individual's Transcript
4. Who spoke the most? least?
5. Headline per Participant

In [ ]:
without_most_talk

2. Number of times each participant spoke.

In [ ]:
def times_each_spoke(times_each_spoke):
    # How many times did each person speak?
    times_each_spoke = pd.crosstab(df['speaker'], columns = 'speaker')
    times_each_spoke = times_each_spoke['speaker'].sort_values(ascending = False)
#     times_each_spoke = times_each_spoke.drop(['UNK'])
    return times_each_spoke

In [ ]:
times_each_spoke(times_each_spoke)

# What did they say?

Wordclouds

# Count the order of speaking as a hover or speaking

In [ ]:
# wordcloud_summary()

In [ ]:
# df['branch'] = df.groupby(['speaker'])['text'].transform(lambda x : ' '.join(x))
# wordcloud = df[['speaker', 'branch']].drop_duplicates()
# UNK = wordcloud[wordcloud['speaker'] == 'UNK'].index
# wordcloud.drop(UNK, inplace= True)


# participants = []
# participant_transcript = []
# for i,j in zip(wordcloud['branch'],wordcloud['speaker']):
#         print(f"TRANSCRIPT Person {j}:")
#         print(i)
#         participant_transcript.append(i)
#         participants.append(j)

#         # mask = np.array(Image.open('Man_Silhouette.jpeg'))
#         wc = WordCloud(stopwords=STOPWORDS,
# #                        mask=mask, 
#                        background_color="white",
#                        max_words=4000, 
#                        max_font_size=256,
#                        random_state=42, 
# #                        width=mask.shape[1],
# #                        height=mask.shape[0], 
#                        contour_width = 1.5, 
#                        colormap = 'Blues')
#         # wc.generate(i)
#         # plt.imshow(wc, interpolation="bilinear")
#         plt.axis('off')
#         # plt.savefig('man_cloud.png', dpi = 500)
#         plt.show()
#         # print('\n')

# What words did each person use? # Can you clean it up? Keywords, counted?

In [ ]:
nlp = spacy.load("en_core_web_sm")

for i,j in zip(participant_transcript, participants):
    text = i
    doc = nlp(text)
    print("\n")
    print(F"Here are the words Person {j} used the most:")

    common_entities = []
    for entity in doc.ents:
        common_entities.append(entity.text)
#         print(entity.text, f" - ({entity.label_})")
        duplicate_dict = {i:common_entities.count(i) for i in common_entities}
    duplicate_dict

    sorted_tuples = sorted(duplicate_dict.items(), key=lambda item: item[1], reverse=True)

    sorted_dict = {k: v for k, v in sorted_tuples}
    for i in sorted_dict:
        print(f"- {i}")  # {1: 1, 3: 4, 2: 9}


# Which Bigrams/Trigrams did each person use?

***Sanitized version***

***Colloquial Version***

In [ ]:
for i,j in zip(participant_transcript, participants):
    text = i
    doc = nlp(text)
    
    def extract_ngrams(data, num):
        n_grams = ngrams(nltk.word_tokenize(data), num)
        return [ ' '.join(grams) for grams in n_grams]

    bi_wrd_phrase = i
    bi_wrd_phrase = extract_ngrams(bi_wrd_phrase,2)
    bi_wrd_phrase = pd.value_counts((bi_wrd_phrase)).head(5)

    tri_wrd_phrase = i
    tri_wrd_phrase = extract_ngrams(tri_wrd_phrase,3)
    tri_wrd_phrase = pd.value_counts((tri_wrd_phrase)).head(5)

    print(F"PERSON {j}:")
    print(bi_wrd_phrase)
    print("\n")  
    print(tri_wrd_phrase)
    print("\n")

# Google Search based on Keywords

In [ ]:
# from googlesearch import search

# print('3 suggested articles based on conversation keywords') 
# for j in search(trigrams, num_results=2):
#     print(j)
# #         print("Articles based on your words")
# #         print(trigrams)    

# How similar were each person's statements to each other?

# Who did they speak with?

Chord Diagram

In [ ]:
type(names)

In [ ]:
type(matrix)

In [ ]:
# names_label.values.tolist()

In [ ]:
names_label.values

In [ ]:
plt_matrix = [
    [0, 5, 6, 4, 7, 4],
    [5, 0, 5, 4, 6, 5],
    [6, 5, 0, 4, 5, 5],
    [4, 4, 4, 0, 5, 5],
    [7, 6, 5, 5, 0, 4],
    [4, 5, 5, 5, 4, 0],
]

names = ["Action", "Adventure", "Comedy", "Drama", "Fantasy", "Thriller"]

Chord(plt_matrix,names).show()

In [ ]:
type(plt_matrix[2][0])

In [ ]:
plt_matrix[2][0]

In [ ]:
# type(matrix[2][0])

In [ ]:
# names_label

In [ ]:
# chord_diagram()

# Summarization per participant

In [ ]:
%timeit
from sumy.parsers.html import HtmlParser
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer as Summarizer
from sumy.nlp.stemmers import Stemmer
from sumy.utils import get_stop_words

In [ ]:
LANGUAGE = "english"
SENTENCES_COUNT = 3

In [ ]:
for i,j in zip(participant_transcript, participants):
    parser = PlaintextParser.from_string(i, Tokenizer(LANGUAGE))
    stemmer = Stemmer(LANGUAGE)

    summarizer = Summarizer(stemmer)
    summarizer.stop_words = get_stop_words(LANGUAGE)
    sentence = summarizer(parser.document, SENTENCES_COUNT)
    
    print(f"HEADLINE: Person {j}")
    print(f"'{sentence[0]}'")
    print("\n")

***Most questions + high idea density = ?***

In [ ]:
#Number of unique words

# Who uses questions like 'Are there any more questions?'

***Any questions***

In [ ]:
# Real participant speech was removed from this notebook.
# Put sample text in a file under media/ (which git ignores) and load it here.
from pathlib import Path
_sample = Path('media/sample_dialogue.txt')
joel_dialogue = _sample.read_text(encoding='utf-8') if _sample.is_file() else ''


In [ ]:
import nltk   
joel_list = nltk.sent_tokenize(joel_dialogue)

In [ ]:
joel_list

In [ ]:
part_list = ' '.join(participant_transcript)
part_list = nltk.sent_tokenize(part_list)
# part_list

In [ ]:
#find all the sentences that are "some"
new_list =[]
new_participant = []

for x in part_list: 
#     if ("to be honest") in x:
    if ("?" in x) & ("how " in x):
        new_list.append(x)
for i in new_list:
    print(i)
# print(new_list)

In [ ]:
# for i in part_list:
#     print(i)

In [ ]:
#find all the sentences that are "some"
new_list =[]
for x in joel_list: 
#     if ("to be honest") in x:
    if ("?" in x) & (' ' in x):
    
        new_list.append(x)
for i in new_list:
    print(i)
# print(new_list)

In [ ]:
# for i in new_list:
#     print(i)

***Some questions***

In [ ]:
#find all the sentences that are "some"
new_list =[]
for x in joel_list: 
#     if ("to be honest") in x:
    if ("?" in x) & (' any' in x):
    
        new_list.append(x)
# print(new_list)

# Send in email

# https://www.youtube.com/watch?v=K7ng0dN2GKo